# 05 — Evaluation and Benchmark

CNN vs radiomics Random Forest, with an optional frozen pathology-foundation-model linear probe. All arms use slide-level holdout.

In [ ]:
from utils import st_helpers as st

ROOT, PHARMA = st.setup_pharma_paths()
st.set_seeds()
print('ROOT:', ROOT)
print('PHARMA:', PHARMA)


In [ ]:
import pandas as pd
from src.data import load_config, cohort_slide_ids
from src.labels import build_labels_cohort
from src.benchmark import run_and_save_benchmark
from src.eval import load_benchmark_report

cfg = load_config()
# Optional research-only arm (Kaiko non-commercial weights):
# cfg['foundation']['enabled'] = True
oncology = cfg['cohorts']['oncology']
labels = build_labels_cohort(cohort_slide_ids(cfg), cfg=cfg)
breast_labels = labels[labels['slide_id'].isin(oncology)]
report_path, results = run_and_save_benchmark(oncology, breast_labels, cfg=cfg)
load_benchmark_report(report_path, cfg=cfg)


### Frozen foundation-model arm

When `cfg['foundation']['enabled']` is true, the benchmark downloads the compact pathology encoder once, runs it strictly in inference mode, caches one embedding array per slide, and fits only logistic/ridge probes within each LOSO fold. The encoder receives no gradient updates.

**License:** the tutorial-sized `kaiko_vits16` checkpoint is non-commercial. Do not use its output for commercial pharma decisions.

In [ ]:
from src.eval import predict_cnn
from src.patches import load_patch_arrays

for sid in cfg['cohorts']['external']:
    try:
        patches, _ = load_patch_arrays(sid, cfg=cfg)
        predict_cnn(results[-1]['model'], patches[:50], device=results[-1]['device'])
        print(sid, 'external OK, spots sampled: 50')
    except FileNotFoundError as e:
        print(sid, e)


**Next:** `06_interpretability.ipynb`